# Puntos y cajas — etapa 1: juego y validación
Esta entrega implementa las reglas. Todavía no contiene entrenamiento ni un modelo aprendido.

Guarda este notebook y `juego.py` en la misma carpeta. Ejecuta las celdas de arriba hacia abajo. Se requiere Python 3 y un entorno Jupyter (por ejemplo, VS Code con soporte para notebooks).

M representa el lado del futuro modelo; J, el jugador aleatorio. En esta primera prueba **ambos eligen líneas libres al azar**. Cerrar una caja da un punto y conserva el turno; cerrar dos da dos puntos. Tras ocupar las 12 líneas, gana quien tiene más cajas; puede haber empate.

`juego.py` contiene constantes y la clase `Juego`. Importarlo no inicia partidas. El notebook dirige las jugadas e imprime el texto que devuelve el juego. No hay entrada humana.

In [17]:
import random
from juego import Juego, MODELO, JUGADOR, SIMBOLOS, EXTREMOS

# Semilla para poder repetir esta demostración.
random.seed(40)

## Acciones y estado
Los puntos se numeran por filas: 0, 1, 2 arriba; 3, 4, 5 al centro; 6, 7, 8 abajo. Una acción es el índice de una línea, sin dirección.

El juego guarda una lista de 12 booleanos para líneas y otra de 4 propietarios de cajas (0 vacía, 1 M, 2 J), además del turno. Las tuplas constantes indican qué líneas forman cada caja. No se necesita una biblioteca de grafos.

`obtener_estado()` entrega una instantánea completa e inmutable. La codificación para el aprendizaje y las recompensas se definirán en una etapa posterior.

In [18]:
for accion, extremos in enumerate(EXTREMOS):
    print(f'Acción {accion:2}: puntos {extremos[0]}–{extremos[1]}')

juego = Juego(jugador_inicial=MODELO)
print(juego.representar())
print('Líneas libres:', juego.movimientos_validos())

Acción  0: puntos 0–1
Acción  1: puntos 1–2
Acción  2: puntos 3–4
Acción  3: puntos 4–5
Acción  4: puntos 6–7
Acción  5: puntos 7–8
Acción  6: puntos 0–3
Acción  7: puntos 1–4
Acción  8: puntos 2–5
Acción  9: puntos 3–6
Acción 10: puntos 4–7
Acción 11: puntos 5–8
 +    0    +    1    + 
 6         7         8 
 +    2    +    3    + 
 9         10        11
 +    4    +    5    + 
M: 0 | J: 0
Turno: M
Líneas libres: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## Elección aleatoria y control de la partida
`random.choice` elige uniformemente entre las líneas libres. Consultamos el turno antes de cada jugada; no lo alternamos desde fuera, porque puede haber capturas consecutivas.

El parámetro `mostrar` permite observar cada paso o ejecutar partidas silenciosas. Si omitimos `jugador_inicial`, la clase sortea quién empieza.

In [19]:
def elegir_accion_aleatoria(juego):
    return random.choice(juego.movimientos_validos())


def ejecutar_partida(jugador_inicial=None, mostrar=True):
    juego = Juego(jugador_inicial)
    if mostrar:
        print(juego.representar())
    while not juego.terminado():
        jugador = juego.obtener_turno()
        # En esta etapa ambos lados son aleatorios.
        # Aquí se incorporará luego la elección del modelo cuando juegue M.
        accion = elegir_accion_aleatoria(juego)
        resultado = juego.jugar(accion)
        if mostrar:
            print(f'\n{SIMBOLOS[jugador]} elige línea {accion}; '
                  f'captura {len(resultado["cajas_capturadas"])} caja(s).')
            print(juego.representar())
    return juego

In [20]:
partida = ejecutar_partida(jugador_inicial=MODELO)

 +    0    +    1    + 
 6         7         8 
 +    2    +    3    + 
 9         10        11
 +    4    +    5    + 
M: 0 | J: 0
Turno: M

M elige línea 7; captura 0 caja(s).
 +    0    +    1    + 
 6         |         8 
 +    2    +    3    + 
 9         10        11
 +    4    +    5    + 
M: 0 | J: 0
Turno: J

J elige línea 10; captura 0 caja(s).
 +    0    +    1    + 
 6         |         8 
 +    2    +    3    + 
 9         |         11
 +    4    +    5    + 
M: 0 | J: 0
Turno: M

M elige línea 9; captura 0 caja(s).
 +    0    +    1    + 
 6         |         8 
 +    2    +    3    + 
 |         |         11
 +    4    +    5    + 
M: 0 | J: 0
Turno: J

J elige línea 0; captura 0 caja(s).
 + ------- +    1    + 
 6         |         8 
 +    2    +    3    + 
 |         |         11
 +    4    +    5    + 
M: 0 | J: 0
Turno: M

M elige línea 4; captura 0 caja(s).
 + ------- +    1    + 
 6         |         8 
 +    2    +    3    + 
 |         |         11
 + ------- + 

## Comprobaciones de las reglas
Un `assert` verifica una condición: si no se cumple, detiene la celda con un error. Estas pruebas cubren captura simple y doble, continuidad del turno, rechazo de acciones sin modificar el estado y copias independientes.

In [21]:
prueba = Juego(MODELO)
for accion in (0, 2, 6):
    prueba.jugar(accion)
jugador = prueba.obtener_turno()
resultado = prueba.jugar(7)
assert resultado['cajas_capturadas'] == (0,)
assert prueba.obtener_turno() == jugador
assert prueba.puntajes()[jugador] == 1

# Dos cajas superiores comparten la línea 7.
doble = Juego(MODELO)
for accion in (0, 2, 6, 1, 3, 8):
    doble.jugar(accion)
jugador = doble.obtener_turno()
resultado = doble.jugar(7)
assert resultado['cajas_capturadas'] == (0, 1)
assert doble.puntajes()[jugador] == 2
assert doble.obtener_turno() == jugador

for invalida in (7, -1, 12, 1.5, True, '3'):
    antes = doble.obtener_estado()
    try:
        doble.jugar(invalida)
    except ValueError:
        pass
    else:
        raise AssertionError('Se aceptó una acción inválida')
    assert doble.obtener_estado() == antes

copia = doble.copiar()
antes = doble.obtener_estado()
copia.jugar(copia.movimientos_validos()[0])
assert doble.obtener_estado() == antes
assert copia.obtener_estado() != antes
print('Capturas, turnos, validaciones y copia: correctos.')

Capturas, turnos, validaciones y copia: correctos.


## Validación con 1.000 partidas
Alternamos quién empieza para probar ambos casos. Cada partida debe consumir exactamente 12 acciones, repartir cuatro cajas y finalizar sin turno pendiente. Estos resultados comprueban el juego; **no miden aprendizaje**, porque todavía no entrenamos un agente.

In [22]:
total_partidas = 1000

conteo = {0: 0, MODELO: 0, JUGADOR: 0}
suma_cajas_m = 0
suma_diferencias = 0

# Resultados separados según quién comienza.
por_inicio = {
    MODELO: {0: 0, MODELO: 0, JUGADOR: 0},
    JUGADOR: {0: 0, MODELO: 0, JUGADOR: 0},
}

for numero in range(total_partidas):
    # Alternamos el inicio para evaluar ambos casos por igual.
    inicial = MODELO if numero % 2 == 0 else JUGADOR
    juego = Juego(inicial)
    pasos = 0

    while not juego.terminado():
        jugador = juego.obtener_turno()
        accion = elegir_accion_aleatoria(juego)
        resultado = juego.jugar(accion)
        pasos += 1

        assert len(juego.movimientos_validos()) == 12 - pasos

        if not juego.terminado():
            if resultado['cajas_capturadas']:
                assert juego.obtener_turno() == jugador
            else:
                assert juego.obtener_turno() != jugador

    puntos = juego.puntajes()
    ganador = juego.ganador()

    assert pasos == 12
    assert sum(puntos.values()) == 4
    assert juego.obtener_turno() is None

    # Acumulamos los resultados para calcular las métricas.
    conteo[ganador] += 1
    por_inicio[inicial][ganador] += 1
    suma_cajas_m += puntos[MODELO]
    suma_diferencias += puntos[MODELO] - puntos[JUGADOR]

    # Una partida terminada debe rechazar nuevas acciones.
    antes = juego.obtener_estado()
    try:
        juego.jugar(0)
    except ValueError:
        pass
    else:
        raise AssertionError('Se permitió jugar tras el final')

    assert juego.obtener_estado() == antes

print(f'Partidas verificadas: {total_partidas}')
print('Evaluación inicial: ambos participantes juegan al azar.')

print('\nResultados desde la perspectiva de M:')
print(f'Victorias: {conteo[MODELO]} '
      f'({100 * conteo[MODELO] / total_partidas:.2f}%)')
print(f'Empates:   {conteo[0]} '
      f'({100 * conteo[0] / total_partidas:.2f}%)')
print(f'Derrotas:  {conteo[JUGADOR]} '
      f'({100 * conteo[JUGADOR] / total_partidas:.2f}%)')

print(f'\nCajas promedio de M: {suma_cajas_m / total_partidas:.3f}')
print(f'Diferencia promedio de cajas (M - J): '
      f'{suma_diferencias / total_partidas:.3f}')

for inicial, resultados in por_inicio.items():
    cantidad = sum(resultados.values())

    if cantidad > 0:
        print(f'\nCuando inicia {SIMBOLOS[inicial]} '
              f'({cantidad} partidas):')
        print(f'  Victorias M: '
              f'{100 * resultados[MODELO] / cantidad:.2f}%')
        print(f'  Empates:     '
              f'{100 * resultados[0] / cantidad:.2f}%')
        print(f'  Derrotas M:  '
              f'{100 * resultados[JUGADOR] / cantidad:.2f}%')

Partidas verificadas: 1000
Evaluación inicial: ambos participantes juegan al azar.

Resultados desde la perspectiva de M:
Victorias: 409 (40.90%)
Empates:   168 (16.80%)
Derrotas:  423 (42.30%)

Cajas promedio de M: 1.986
Diferencia promedio de cajas (M - J): -0.028

Cuando inicia M (500 partidas):
  Victorias M: 39.80%
  Empates:     17.80%
  Derrotas M:  42.40%

Cuando inicia J (500 partidas):
  Victorias M: 42.00%
  Empates:     15.80%
  Derrotas M:  42.20%


## Próxima etapa
Elegiremos el algoritmo aprobado para el examen, la información que recibirá el agente y las recompensas. Después añadiremos entrenamiento, evaluación y persistencia en este mismo notebook.

Para estudiar `juego.py`: revisar constantes, constructor, consultas, copias, puntajes, final de partida, aplicación de acciones y representación. Los comentarios se mantienen para aprender; podrás retirarlos cuando prepares la defensa.